In [2]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import os
# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

# Configurae the sampling parameters (for thinking mode)
sampling_params = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=32768)
os.environ.pop("TRITON_PTXAS_PATH", None)

# Initialize the vLLM engine
llm = LLM(
    model="Qwen/Qwen3-8B",
    dtype="float16",   # 或 "half"
    max_num_seqs=8,
    max_model_len=2048,
    gpu_memory_utilization=0.7,  # 預設 ~0.9 → 改小
    enforce_eager=True,
    enable_chunked_prefill=False,
)

# Prepare the input to the model
prompt = "Give me a short introduction to large language models."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,  # Set to False to strictly disable thinking
)

# Generate outputs
outputs = llm.generate([text], sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")


/home/u1501463/miniconda3/envs/qwen3_omni/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-23 12:09:00 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 04-23 12:09:06 [__init__.py:239] Automatically detected platform cuda.


2026-04-23 12:09:17,699	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


WARNING 04-23 12:09:22 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 04-23 12:09:41 [config.py:717] This model supports multiple tasks: {'generate', 'reward', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
WARNING 04-23 12:09:41 [arg_utils.py:1658] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 04-23 12:09:41 [cuda.py:93] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 04-23 12:09:41 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_red

AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended